[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/intisariapps-com/Intisari-AutoCut-Android/blob/main/AutoCut_Video_Engine_Colab.ipynb)

# 🎬 AutoCut Video AI — Server Komputasi Cloud Colab

> 🚀 **Pusat Komputasi Cloud:** Memproses render video vertikal beresolusi tinggi dengan akselerasi GPU & RAM-Disk
> ⚡ **Cloudflare R2 Global CDN:** Distribusi video ultra-cepat langsung ke galeri HP tanpa leher botol tunnel
> 🛡️ **Anti-Timeout & Asynchronous:** Antrean pekerjaan otomatis yang stabil dan tahan gangguan jaringan
> 📱 **Scan-to-Pairing:** Koneksi instan 1-detik dari aplikasi Android via pemindaian QR Code

Notebook ini berfungsi sebagai **Engine Backend Rendering Video Klip** untuk aplikasi **Intisari AutoCut Android**.

### ✨ Fitur Unggulan Sistem:
1. 📦 **Batch Recipe Serverless**: Unduh master video YouTube 1x, proses otomatis puluhan klip resep sekaligus.
2. 🌐 **Cloudflare R2 High-Speed CDN**: Klip video hasil render langsung dialirkan via Anycast CDN edge berkecepatan tinggi.
3. ⚡ **Producer-Consumer Streaming**: Begitu 1 klip selesai dirender di server, aplikasi Android langsung mengunduhnya secara paralel.
4. 🎙️ **Sub-Second Groq Whisper LPU**: Transkripsi audio kilat tingkat kata (<0.8s per klip) tanpa membebani GPU lokal.
5. 🖥️ **MediaPipe Dynamic Face Tracking**: Pelacakan wajah otomatis berbasis AI untuk reframe vertikal 9:16 yang presisi.
6. 🎨 **Preset Visual Hook Card**: Kartu headline visual otomatis (Breaking News, TikTok Card, Neon, dll.) dengan Pillow RGBA.
7. 🧠 **Model AI Lokal GGUF On-GPU (Qwen2.5)**: Kurasi transkrip klip viral 100% bebas API Key langsung di GPU Colab.
8. 📱 **Scan-to-Pairing (QR Code)**: Tautan server otomatis dikonversi ke gambar QR Code untuk pairing kamera instan dari HP.
9. ⏱️ **Live Countdown Watchdog**: Auto-shutdown runtime saat idle untuk menghemat kuota pemakaian GPU.

---
### 🚀 Cara Menjalankan:
1. Klik tombol **Play (▶)** di sel kode di bawah ini.
2. Tunggu hingga proses setup selesai dan **Tautan Server**, **Spesifikasi Hardware**, serta **QR Code** muncul di layar.
3. Buka aplikasi **Intisari AutoCut Android** di HP Anda -> Buka Tab **Pengaturan** -> Pindai QR Code atau masukkan tautan tersebut.


In [ ]:
"""
🎬 AUTOCUT VIDEO ENGINE — SERVER RENDERING & CLOUDFLARE TUNNEL
Hak Cipta (C) 2026 IntisariApps.com. Seluruh hak cipta dilindungi.
"""

# @title ⚙️ PUSAT KENDALI ENGINE RENDERING COLAB
# @markdown Atur parameter sesi Colab di bawah ini:
AUTO_SHUTDOWN_MINUTES = 10  # @param [0, 5, 10, 15, 30] {type:"raw"}
GOOGLE_DRIVE_SYNC = "oauth_persistent"  # @param ["oauth_persistent", "native_mount", "oauth_api", "disabled"]
GDRIVE_FOLDER_NAME = "AutoCut_Videos"  # @param {type:"string"}
GDRIVE_TOKEN_JSON = ""  # @param {type:"string"}
COOKIE_SOURCE = "gdrive"  # @param ["gdrive", "legacy", "disabled"]
GROQ_API_KEY = ""  # @param {type:"string"}
GEMINI_PSID = ""  # @param {type:"string"}
GEMINI_PSIDTS = ""  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}

import os
import sys
import time
import re
import json
import shutil
import zipfile
import subprocess
import sysconfig
import threading
import urllib.request

if "COOKIE_SOURCE" in globals():
    os.environ["COOKIE_SOURCE"] = str(COOKIE_SOURCE).lower().strip()

print("=" * 80)
print("🚀 MEMULAI AUTOCUT VIDEO ENGINE (BYOC SERVER)")
print("=" * 80)


# Pastikan mount Drive Saya tersedia untuk AutoCut_Studio & Cookies
if "COOKIE_SOURCE" in globals() and str(COOKIE_SOURCE).lower() == "gdrive":
    os.environ["COOKIE_SOURCE"] = "gdrive"
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            print("📁 [Google Drive] Menghubungkan Drive Saya untuk folder AutoCut_Studio...", flush=True)
            from google.colab import drive
            drive.mount("/content/drive")
            print("✅ [Google Drive] Drive Saya terhubung.", flush=True)
        except Exception as e_drv_mnt:
            print(f"⚠️ [Google Drive] Native mount: {e_drv_mnt}", flush=True)

# 0. Penanganan integrasi Google Drive (Auto-Mount untuk Cookies & Clips)
if ("COOKIE_SOURCE" in globals() and str(COOKIE_SOURCE).lower() == "gdrive") or (GOOGLE_DRIVE_SYNC in ("native_mount", "oauth_persistent")):
    os.environ["COOKIE_SOURCE"] = "gdrive"
    print("📁 [Google Drive] Menghubungkan Drive Saya untuk folder AutoCut_Studio...", flush=True)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("✅ [Google Drive] Drive Saya terhubung via native mount.", flush=True)
    except Exception as e_mnt:
        print(f"⚠️ [Google Drive] Native mount: {e_mnt}", flush=True)

    # LANGSUNG BUAT FOLDER SECARA NATIVE (Zero-Dependency & Fail-Fast di detik pertama)
    drive_base = "/content/drive/MyDrive"
    if os.path.exists(drive_base):
        studio_root = os.path.join(drive_base, "AutoCut_Studio")
        yt_dir = os.path.join(studio_root, "Cookies", "youtube")
        gemini_dir = os.path.join(studio_root, "Cookies", "gemini")
        clips_dir = os.path.join(studio_root, "Clips")
        
        os.makedirs(yt_dir, exist_ok=True)
        os.makedirs(gemini_dir, exist_ok=True)
        os.makedirs(clips_dir, exist_ok=True)
        
        readme_file = os.path.join(studio_root, "Cookies", "README_PANDUAN.txt")
        if not os.path.exists(readme_file):
            with open(readme_file, "w", encoding="utf-8") as rf:
                rf.write("# PANDUAN PENGGUNAAN FOLDER COOKIE AUTOCUT STUDIO\n\n"
                         "Folder ini menyimpan sesi akun YouTube & Gemini secara persisten di Google Drive Anda.\n"
                         "1. youtube/ -> Letakkan file cookie Netscape YouTube (youtube_cookies.txt)\n"
                         "2. gemini/  -> Letakkan file cookie Gemini Web (akun_01.json atau akun_01.txt)\n")
        print("🎉 [Google Drive] Folder Master 'AutoCut_Studio' (Cookies & Clips) BERHASIL DIBUAT di Drive Saya!", flush=True)
    else:
        print("⚠️ [Google Drive] Direktori /content/drive/MyDrive belum terbaca di sistem Colab.", flush=True)

print("=" * 80)
    print(f"👉 URL TUNNEL PUBLIK : {public_tunnel_url}")
    print("-" * 80)
    print("🖥️  PROFIL SPESIFIKASI SERVER COLAB:")
    print(f"   • GPU Hardware       : {gpu_name} ({gpu_vram} MB VRAM)")
    print(f"   • CPU Processor      : {cpu_model} ({cpu_cores} Cores)")
    print(f"   • RAM Sistem         : {ram_total} MB")
    print(f"   • RAM-Disk (/dev/shm): {ram_disk_mb} MB (Buffer Cepat 4.000 MB/s)")
    print(f"   • Encoder Video      : {encoder.upper()} (Akselerasi Aktif)")
    print(f"   • Estimasi Kecepatan : {benchmark}")
    print("=" * 80)

    # Tampilkan QR Code di layar Colab untuk pairing kamera smartphone
    try:
        import qrcode
        from IPython.display import display, Image
        import io
        qr = qrcode.QRCode(box_size=7, border=2)
        qr.add_data(public_tunnel_url)
        qr.make(fit=True)
        img = qr.make_image(fill_color="black", back_color="white")
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        print("\n📱 PINDAI QR CODE DI BAWAH INI DARI APLIKASI ANDROID:")
        display(Image(buf.getvalue()))
    except Exception as e_qr:
        print(f"(QR Code viewer fallback: {e_qr})")

    print("\nℹ️ Server siap menerima instruksi render video dari aplikasi Android!")
    print("⏱️ Tekan tombol Stop (⏹) kapan saja untuk mematikan server.")

# 6. Live Interactive Watchdog Timer (ATM dari intiVoice V1.6.3)
def _idle_watchdog_loop():
    try:
        from autocut_video_engine.server import watchdog
    except ImportError:
        try:
            from autocut_video_engine.watchdog import watchdog
        except ImportError:
            from colab.engine.watchdog import watchdog
    timeout_min = float(AUTO_SHUTDOWN_MINUTES)
    if timeout_min <= 0:
        print("⏱️ [AUTO-SHUTDOWN NONAKTIF] Mesin akan standby tanpa batas waktu.")
        while True:
            time.sleep(1)
        return

    idle_limit_sec = timeout_min * 60.0
    display_handle = None
    try:
        from IPython.display import display, HTML
        init_html = (
            "<div style='font-family: monospace; font-size: 13px; color: #00D2B4; background: #071952; "
            "padding: 10px 16px; border-radius: 10px; border: 1px solid #1A73E8; margin-top: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);'>"
            "⏳ <b>COUNTDOWN AUTO-SHUTDOWN:</b> Menyiapkan timer realtime..."
            "</div>"
        )
        display_handle = display(HTML(init_html), display_id=True)
    except Exception:
        display_handle = None

    while True:
        time.sleep(1.0)
        try:
            is_active = getattr(watchdog, "is_busy", getattr(watchdog, "active_jobs", 0) > 0)
            if is_active:
                watchdog.touch()

            elapsed_idle = time.time() - watchdog.last_activity
            remaining_sec = max(0, int(idle_limit_sec - elapsed_idle))
            mins = remaining_sec // 60
            secs = remaining_sec % 60

            if display_handle:
                try:
                    if is_active:
                        widget_html = (
                            "<div style='font-family: monospace; font-size: 13px; color: #FFD700; background: #1a1a00; "
                            "padding: 10px 16px; border-radius: 10px; border: 1px solid #FFA500; margin-top: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);'>"
                            "⚡ <b>ENGINE SEDANG MERENDER:</b> Memproses video klip... <span style='color: #00D2B4;'>(Timer idle dijeda)</span>"
                            "</div>"
                        )
                    else:
                        badge_color = "#00D2B4" if remaining_sec > 60 else ("#FFA500" if remaining_sec > 30 else "#FF4444")
                        bg_color = "#071952" if remaining_sec > 60 else ("#2b1700" if remaining_sec > 30 else "#2b0000")
                        border_color = "#1A73E8" if remaining_sec > 60 else ("#FF8C00" if remaining_sec > 30 else "#FF0000")

                        widget_html = (
                            f"<div style='font-family: monospace; font-size: 13px; color: {badge_color}; background: {bg_color}; "
                            f"padding: 10px 16px; border-radius: 10px; border: 1px solid {border_color}; margin-top: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);'>"
                            f"⏳ <b>COUNTDOWN AUTO-SHUTDOWN:</b> Sisa Waktu Idle: <b style='font-size: 16px; color: #FFFFFF;'>{mins:02d}:{secs:02d}</b> "
                            f"<span style='font-size: 11px; opacity: 0.8;'>| Reset otomatis tiap ada job render baru</span>"
                            f"</div>"
                        )
                    from IPython.display import HTML
                    display_handle.update(HTML(widget_html))
                except Exception:
                    pass

            if elapsed_idle >= idle_limit_sec and not is_active:
                print(f"\n🛑 [AUTO-SHUTDOWN] TIDAK ADA AKTIVITAS RENDER SELAMA {int(timeout_min)} MENIT.")
                print("💡 Memutuskan runtime Google Colab untuk menghemat kuota compute units...")
                try:
                    from google.colab import runtime
                    runtime.unassign()
                except Exception:
                    os._exit(0)
                return
        except Exception:
            pass

try:
    _idle_watchdog_loop()
except KeyboardInterrupt:
    print("\n🛑 Sesi server dihentikan oleh pengguna.")
